# UniCell Annotation of Large Datasets in Sample-Level Chunks

This notebook is designed for large `.h5ad` datasets that cannot be loaded into memory in full. It opens the data in backed mode and loads, predicts, and saves **one sample at a time** according to `adata.obs[SAMPLE_KEY]`.

Each sample generates:

- An `.h5ad` file containing the complete prediction results
- A `.csv` file containing only the main predicted labels
- An optional JSON file containing Accuracy / Macro-F1 metrics

The run status of all samples is continuously written to `chunk_summary.csv`.

> **Important**: Here, a chunk is one complete sample, not a fixed number of cells. If a sample is itself very large, memory may still be insufficient. Run the preview first and check the number of cells in each sample.

> For small or medium-sized datasets, use `predict.ipynb` in the same directory.

## Step 1: Load the Runtime Environment

If a required Python package is reported missing here, contact the model maintainer to configure the UniCell runtime environment.

In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import time

import anndata as ad
import numpy as np
import pandas as pd
import torch

from unicell.anno_predict import unicell_predict
from unicell.utils.utils import compute_metrics as compute_metrics_cls

print("Runtime environment loaded successfully")
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Step 2: Specify Paths and Parameters

The following settings generally need to be reviewed or modified:

- `INPUT_H5AD`: The large `.h5ad` file to annotate.
- `OUTPUT_DIR`: The directory in which to save the results for each sample.
- `SAMPLE_KEY`: The field in `adata.obs` used to distinguish samples. The default is `sample`.
- `PREVIEW_ONLY=True`: Inspect the dataset size without running the model. Keeping this set to `True` is recommended for the first run.
- `SELECTED_SAMPLES`: Run only the specified samples; an empty list means all samples.
- `MAX_SAMPLES`: The maximum number of samples to run. Setting this to `1` is recommended for the initial test.
- `SKIP_DONE=True`: Skip a sample if both its `.h5ad` and `.csv` files already exist, enabling interrupted runs to resume.
- `BATCH_SIZE`: Reduce this to 1024, 512, or lower if GPU memory is insufficient.

The following settings are recommended for the first run:

```python
PREVIEW_ONLY = True
MAX_SAMPLES = 1
```

In [ ]:
# ===== Path settings =====
PROJECT_DIR = Path.cwd()
INPUT_H5AD = Path("/path/to/input.h5ad")
MODEL_DIR = PROJECT_DIR / "models/last_version_gml"
OUTPUT_DIR = PROJECT_DIR / "outputs/chunks"
SUMMARY_CSV = OUTPUT_DIR / "chunk_summary.csv"

# ===== Chunking settings =====
SAMPLE_KEY = "sample"
PREVIEW_ONLY = True          # Set to False after checking the preview to run the model
SELECTED_SAMPLES = []        # For example, ["sample_01", "sample_02"]; an empty list selects all samples
MAX_SAMPLES = 1              # Recommended: 1 for the first test; set to None to process all samples
SKIP_DONE = True             # Recommended: keep True to allow interrupted runs to resume

# ===== Model and output settings =====
DEVICE = "auto"            # "auto", "cuda", or "cpu"
BATCH_SIZE = 2048
USE_SAFE_LIST = True
SAFE_LIST_PATH = PROJECT_DIR / "safe_list.json"
OUTPUT_COMPRESSION = "gzip"  # "gzip" or None
COMPRESSION_LEVEL = 4        # gzip compression level: 0-9
SAVE_METRICS_JSON = True

# Ground-truth label fields in the input data; evaluation is skipped when ground-truth labels are unavailable
CELL_TYPE_KEY = "cell_type_ontology_term_id"
TISSUE_KEY = "general_tissue"
SPECIES_KEY = "organism"

### Safe List Example and Notes

The project's `safe_list.json` is a format example. Replace it with a safe list appropriate for the target tissue. The safe list constrains model outputs in the following order:

```text
Allowed species
      ↓
Tissues allowed for that species
      ↓
Cell Ontology IDs allowed for that species+tissue combination
```

Example `safe_list.json`:

```json
{
  "global": {
    "species": ["Homo sapiens", "Mus musculus"],
    "tissue": ["brain", "lung", "heart"],
    "cell_type": ["CL:0000540", "CL:0000127", "CL:0000236"]
  },
  "hierarchy": {
    "Homo sapiens": {
      "tissue": {
        "brain": {"cell_type": ["CL:0000540", "CL:0000127"]},
        "lung": {"cell_type": ["CL:0000236"]}
      }
    },
    "Mus musculus": {
      "tissue": {
        "brain": {"cell_type": ["CL:0000540"]}
      }
    }
  }
}
```

Species and tissue names must exactly match the model's class dictionaries; cell types should be specified using Cell Ontology IDs supported by the model. A safe list can only restrict existing classes and cannot add new classes on which the model was not trained. For non-blood data, disable `USE_SAFE_LIST` or provide a new JSON file appropriate for the experiment.

## Step 3: Validate the Configuration

This cell checks only the paths, model files, and runtime device; it does not load the complete expression matrix.

In [ ]:
if DEVICE == "auto":
    device = "cuda" if torch.cuda.is_available() else "cpu"
elif DEVICE in {"cuda", "cpu"}:
    device = DEVICE
else:
    raise ValueError('DEVICE must be "auto", "cuda", or "cpu".')

if device == "cuda" and not torch.cuda.is_available():
    print("No available GPU detected; automatically switching to CPU.")
    device = "cpu"

required_model_files = [
    "unicell_v1.best.pth",
    "celltype_dict.pk",
    "tissue_dict.pk",
    "species_dict.pk",
    "gene_names.pk",
]
problems = []
if not INPUT_H5AD.is_file():
    problems.append(f"Input file not found: {INPUT_H5AD}")
if INPUT_H5AD.suffix.lower() != ".h5ad":
    problems.append("The input file must have the .h5ad extension")
if not MODEL_DIR.is_dir():
    problems.append(f"Model directory not found: {MODEL_DIR}")
else:
    for filename in required_model_files:
        if not (MODEL_DIR / filename).is_file():
            problems.append(f"Missing from the model directory: {filename}")
if USE_SAFE_LIST and not SAFE_LIST_PATH.is_file():
    problems.append(f"Safe list not found: {SAFE_LIST_PATH}")
if BATCH_SIZE <= 0:
    problems.append("BATCH_SIZE must be greater than 0")
if OUTPUT_COMPRESSION not in {"gzip", None}:
    problems.append('OUTPUT_COMPRESSION must be "gzip" or None')
if OUTPUT_COMPRESSION == "gzip" and not 0 <= COMPRESSION_LEVEL <= 9:
    problems.append("COMPRESSION_LEVEL must be between 0 and 9")
if MAX_SAMPLES is not None and MAX_SAMPLES <= 0:
    problems.append("MAX_SAMPLES must be a positive integer or None")

if problems:
    raise ValueError("Configuration validation failed:\n- " + "\n- ".join(problems))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
safe_list_path = str(SAFE_LIST_PATH) if USE_SAFE_LIST else None

print("Configuration validation passed")
print(f"Input file: {INPUT_H5AD}")
print(f"Model directory: {MODEL_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Grouping field: {SAMPLE_KEY}")
print(f"Device: {device}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Preview only: {PREVIEW_ONLY}")
print(f"Skip completed samples: {SKIP_DONE}")

## Step 4: Open the Data in Backed Mode and Preview the Samples

Backed mode reads only the data structure and `obs`; it does not load the complete expression matrix into memory. The number of cells in each sample is displayed below.

In [ ]:
# If this cell is rerun, first close the previously opened file.
if "adata_backed" in globals() and adata_backed is not None:
    try:
        adata_backed.file.close()
    except Exception:
        pass

open_start = time.perf_counter()
adata_backed = ad.read_h5ad(INPUT_H5AD, backed="r")

if SAMPLE_KEY not in adata_backed.obs.columns:
    available = list(adata_backed.obs.columns)
    adata_backed.file.close()
    adata_backed = None
    raise KeyError(
        f"SAMPLE_KEY='{SAMPLE_KEY}' was not found in obs. First 30 available fields: {available[:30]}"
    )

sample_series = adata_backed.obs[SAMPLE_KEY].astype(str).copy()
unique_samples = pd.Index(sample_series).drop_duplicates().tolist()
sample_counts = (
    sample_series.value_counts(sort=False)
    .rename_axis(SAMPLE_KEY)
    .reset_index(name="Cell count")
)
sample_counts["Percentage of all cells (%)"] = (
    sample_counts["Cell count"] / adata_backed.n_obs * 100
).round(2)
sample_counts = sample_counts.sort_values("Cell count", ascending=False).reset_index(drop=True)

print(f"File opened in {time.perf_counter() - open_start:.1f} seconds")
print(f"Total cell count: {adata_backed.n_obs:,}")
print(f"Total gene count: {adata_backed.n_vars:,}")
print(f"Sample count: {len(unique_samples):,}")
print(f"Largest sample: {sample_counts.iloc[0][SAMPLE_KEY]}, {sample_counts.iloc[0]['Cell count']:,} cells")
display(sample_counts.head(30))

## Step 5: Confirm the Samples to Run

A sample's output number is based on its stable position in the original data. Consequently, later resumed runs use the same filenames even if only a subset of samples is run initially.

In [ ]:
sample_to_chunk_id = {sample: index for index, sample in enumerate(unique_samples, start=1)}

if SELECTED_SAMPLES:
    requested_samples = [str(sample) for sample in SELECTED_SAMPLES]
    missing_samples = [sample for sample in requested_samples if sample not in sample_to_chunk_id]
    if missing_samples:
        raise ValueError(f"The following SELECTED_SAMPLES were not found: {missing_samples}")
    samples_to_run = requested_samples
else:
    samples_to_run = list(unique_samples)

if MAX_SAMPLES is not None:
    samples_to_run = samples_to_run[:MAX_SAMPLES]

run_plan = pd.DataFrame({
    "chunk_id": [sample_to_chunk_id[sample] for sample in samples_to_run],
    "sample": samples_to_run,
    "Cell count": [int((sample_series.values == sample).sum()) for sample in samples_to_run],
})

print(f"Scheduled to process {len(samples_to_run):,} samples")
print(f"Total cell count: {run_plan['Cell count'].sum():,}")
display(run_plan.head(50))

if PREVIEW_ONLY:
    print("PREVIEW_ONLY=True: the model will not run. After checking the plan, set it to False in step 2.")

## Step 6: Define Saving and Evaluation Utilities

This cell only defines the functions required for the run; it does not start inference.

In [ ]:
UNKNOWN_TOKENS = {"unknown", "nan", "None", "NA", "N/A", ""}

def sanitize_filename(value, max_length=120):
    text = re.sub(r'[\\/:*?"<>|\s]+', "_", str(value).strip())
    text = re.sub(r"_+", "_", text).strip("_")
    return (text or "EMPTY")[:max_length]

def evaluate_task(obs, true_key, predicted_key, task_name):
    if true_key not in obs.columns or predicted_key not in obs.columns:
        return None
    pairs = [
        (str(true_value), str(predicted_value))
        for true_value, predicted_value in zip(obs[true_key], obs[predicted_key])
        if str(true_value) not in UNKNOWN_TOKENS
        and str(predicted_value) not in UNKNOWN_TOKENS
    ]
    if not pairs:
        return None
    y_true, y_pred = zip(*pairs)
    raw = compute_metrics_cls(list(y_true), list(y_pred))
    result = {
        "n": len(pairs),
        "accuracy": float(raw.get("accuracy", np.nan)),
        "macro_f1": float(raw.get("macro_f1", np.nan)),
    }
    print(
        f"{task_name}: n={result['n']:,}, "
        f"Accuracy={result['accuracy']:.4f}, Macro-F1={result['macro_f1']:.4f}"
    )
    return result

def compute_metrics_for_adata(adata):
    return {
        "cell_type": evaluate_task(
            adata.obs, CELL_TYPE_KEY, "predicted_cell_type_ontology_id", "Cell type"
        ),
        "tissue": evaluate_task(
            adata.obs, TISSUE_KEY, "predicted_tissue", "Tissue"
        ),
        "species": evaluate_task(
            adata.obs, SPECIES_KEY, "predicted_species", "Species"
        ),
    }

def save_prediction_csv(adata, sample_value, output_csv):
    required = [
        "predicted_cell_type_ontology_id",
        "predicted_tissue",
        "predicted_species",
    ]
    missing = [column for column in required if column not in adata.obs.columns]
    if missing:
        raise KeyError(f"Missing fields in prediction results: {missing}")
    table = pd.DataFrame({
        "obs_name": adata.obs_names.astype(str),
        "predicted_cell_type_ontology_id": adata.obs[required[0]].astype(str).values,
        "predicted_tissue": adata.obs[required[1]].astype(str).values,
        "predicted_species": adata.obs[required[2]].astype(str).values,
        SAMPLE_KEY: (
            adata.obs[SAMPLE_KEY].astype(str).values
            if SAMPLE_KEY in adata.obs.columns
            else str(sample_value)
        ),
    })
    table.to_csv(output_csv, index=False)

def update_summary(row):
    new_row = pd.DataFrame([row])
    if SUMMARY_CSV.is_file():
        try:
            previous = pd.read_csv(SUMMARY_CSV)
        except pd.errors.EmptyDataError:
            previous = pd.DataFrame()
    else:
        previous = pd.DataFrame()
    if not previous.empty and {"chunk_id", "sample"}.issubset(previous.columns):
        same_row = (
            previous["chunk_id"].astype(str).eq(str(row["chunk_id"]))
            & previous["sample"].astype(str).eq(str(row["sample"]))
        )
        previous = previous.loc[~same_row]
    combined = pd.concat([previous, new_row], ignore_index=True, sort=False)
    if "chunk_id" in combined.columns:
        combined = combined.sort_values("chunk_id", kind="stable")
    temporary_csv = SUMMARY_CSV.with_suffix(".csv.tmp")
    combined.to_csv(temporary_csv, index=False)
    os.replace(temporary_csv, SUMMARY_CSV)
    return combined

## Step 7: Start Sample-Level Inference

Inference runs only when `PREVIEW_ONLY=False`. Results are saved and `chunk_summary.csv` is updated immediately after each sample finishes. A failure in one sample does not interrupt the other samples.

> Do not restart the kernel during a long run. Whether closing the browser page terminates the computation depends on the configuration of the Jupyter service being used.

In [ ]:
def run_chunked_inference():
    global adata_backed
    if PREVIEW_ONLY:
        print("PREVIEW_ONLY=True: preview only; the model will not run.")
        print("After checking the sample sizes, set PREVIEW_ONLY=False in step 2.")
        return pd.read_csv(SUMMARY_CSV) if SUMMARY_CSV.is_file() else pd.DataFrame()
    if adata_backed is None:
        raise RuntimeError("The backed data file has been closed; rerun step 4.")

    all_start = time.perf_counter()
    latest_summary = pd.DataFrame()

    try:
        for run_number, sample_value in enumerate(samples_to_run, start=1):
            chunk_id = sample_to_chunk_id[sample_value]
            cell_positions = np.flatnonzero(sample_series.values == sample_value)
            n_cells = int(cell_positions.size)
            sample_tag = sanitize_filename(sample_value)
            prefix = f"pred_{chunk_id:06d}_{sample_tag}"
            output_h5ad = OUTPUT_DIR / f"{prefix}.h5ad"
            output_csv = OUTPUT_DIR / f"{prefix}.csv"
            output_json = OUTPUT_DIR / f"{prefix}_metrics.json"

            print("\n" + "=" * 80)
            print(f"[{run_number}/{len(samples_to_run)}] sample={sample_value}")
            print(f"Cell count: {n_cells:,}")
            print(f"Output: {output_h5ad.name}")

            if SKIP_DONE and output_h5ad.is_file() and output_csv.is_file():
                print("The .h5ad and .csv files already exist; skipping.")
                latest_summary = update_summary({
                    "chunk_id": chunk_id,
                    "sample": str(sample_value),
                    "n_cells": n_cells,
                    "output_h5ad": str(output_h5ad),
                    "output_csv": str(output_csv),
                    "status": "skipped_exists",
                })
                continue

            sub_adata = None
            sc_dataset = None
            try:
                load_start = time.perf_counter()
                sub_adata = adata_backed[cell_positions, :].to_memory()
                load_seconds = time.perf_counter() - load_start
                print(f"Current sample loaded in {load_seconds:.1f} seconds")

                inference_start = time.perf_counter()
                sc_dataset = unicell_predict(
                    adata=sub_adata,
                    filepath=str(INPUT_H5AD),
                    ckpt_dir=str(MODEL_DIR),
                    batch_size=BATCH_SIZE,
                    device=device,
                    cell_type_key=CELL_TYPE_KEY,
                    tissue_key=TISSUE_KEY,
                    species_key=SPECIES_KEY,
                    compute_metrics=False,
                    safe_list_path=safe_list_path,
                )
                inference_seconds = time.perf_counter() - inference_start
                print(f"Model inference: {inference_seconds:.1f} seconds")

                result_adata = sc_dataset.adata
                write_start = time.perf_counter()
                if OUTPUT_COMPRESSION == "gzip":
                    result_adata.write_h5ad(
                        output_h5ad, compression="gzip",
                        compression_opts=COMPRESSION_LEVEL,
                    )
                else:
                    result_adata.write_h5ad(output_h5ad)
                save_prediction_csv(result_adata, sample_value, output_csv)
                write_seconds = time.perf_counter() - write_start
                print(f"Saving h5ad and csv: {write_seconds:.1f} seconds")

                row = {
                    "chunk_id": chunk_id,
                    "sample": str(sample_value),
                    "n_cells": n_cells,
                    "output_h5ad": str(output_h5ad),
                    "output_csv": str(output_csv),
                    "status": "ok",
                    "load_sec": round(load_seconds, 4),
                    "infer_sec": round(inference_seconds, 4),
                    "write_sec": round(write_seconds, 4),
                }

                if SAVE_METRICS_JSON:
                    metrics = compute_metrics_for_adata(result_adata)
                    with output_json.open("w", encoding="utf-8") as file:
                        json.dump(metrics, file, indent=2, ensure_ascii=False)
                    row["metrics_json"] = str(output_json)
                    for task, values in metrics.items():
                        if values is not None:
                            row[f"{task}_accuracy"] = values["accuracy"]
                            row[f"{task}_macro_f1"] = values["macro_f1"]

                latest_summary = update_summary(row)
                print(f"Completed: {sample_value}")

            except Exception as error:
                print(f"Failed: {sample_value}")
                print(f"Reason: {type(error).__name__}: {error}")
                latest_summary = update_summary({
                    "chunk_id": chunk_id,
                    "sample": str(sample_value),
                    "n_cells": n_cells,
                    "output_h5ad": str(output_h5ad),
                    "output_csv": str(output_csv),
                    "status": f"failed: {type(error).__name__}: {error}",
                })
            finally:
                del sub_adata
                del sc_dataset
                gc.collect()
                if device == "cuda":
                    torch.cuda.empty_cache()

    finally:
        try:
            adata_backed.file.close()
        except Exception:
            pass
        adata_backed = None

    print("\n" + "=" * 80)
    print(f"Total runtime: {(time.perf_counter() - all_start) / 60:.1f} minutes")
    print(f"Summary table: {SUMMARY_CSV}")
    return latest_summary

summary = run_chunked_inference()

## Step 8: View the Run Summary

Common values of `status` are:

- `ok`: The sample completed successfully.
- `skipped_exists`: Existing results were detected, so the sample was skipped.
- `failed: ...`: The sample failed, followed by the reason for the error.

In [ ]:
if SUMMARY_CSV.is_file():
    summary = pd.read_csv(SUMMARY_CSV)
    display(summary)
    print("Status counts:")
    display(summary["status"].value_counts(dropna=False).rename("Sample count").to_frame())
else:
    print("chunk_summary.csv has not been generated yet.")

## Step 9: Load the Results of a Completed Sample (Optional)

The following cell loads the first successful result in the summary table and displays the predicted labels. It does not rerun the model.

In [ ]:
if SUMMARY_CSV.is_file():
    summary = pd.read_csv(SUMMARY_CSV)
    successful = summary[summary["status"].eq("ok")]
    if successful.empty:
        print("The summary table does not contain any results with status='ok' yet.")
    else:
        example_path = Path(successful.iloc[0]["output_h5ad"])
        example_adata = ad.read_h5ad(example_path, backed="r")
        columns = [
            column for column in [
                "predicted_cell_type_ontology_id",
                "predicted_tissue",
                "predicted_species",
            ]
            if column in example_adata.obs.columns
        ]
        print(f"Example file: {example_path}")
        print(f"Cell count: {example_adata.n_obs:,}")
        display(example_adata.obs[columns].head(10))
        example_adata.file.close()
else:
    print("chunk_summary.csv has not been generated yet.")

## Output Files

For example, a sample named `CIMA_H001` may generate:

```text
pred_000001_CIMA_H001.h5ad
pred_000001_CIMA_H001.csv
pred_000001_CIMA_H001_metrics.json
```

Where:

- `.h5ad`: Contains the expression matrix, original `obs`, predicted labels, embeddings, and model scores.
- `.csv`: Contains cell IDs, predicted cell types, tissues, species, and samples, making it suitable for direct inspection or import into other software.
- `_metrics.json`: Contains valid evaluation results only when ground-truth labels are available and `SAVE_METRICS_JSON` is enabled.
- `chunk_summary.csv`: Records the status, file locations, elapsed time, and evaluation metrics for every attempted sample.

This notebook does not automatically merge all sample-level `.h5ad` files into one large file. In most cases, the CSV or h5ad file for each sample can be used directly for downstream analysis.

## Frequently Asked Questions

**1. What should I do if GPU memory is insufficient?**  
Reduce `BATCH_SIZE`, for example from 2048 to 1024, 512, or 256.

**2. What should I do if system memory is insufficient?**  
Check the largest sample. The current implementation loads one complete sample at a time; if a single sample is still too large, a second level of chunking by a fixed number of cells is required.

**3. Must I restart from the beginning after an interrupted run?**  
No. Keep `SKIP_DONE=True`, reopen the notebook, and run the cells in order. Samples with existing `.h5ad` and `.csv` files will be skipped.

**4. Why do some samples have a `failed` status?**  
Check the reason in the `status` column of `chunk_summary.csv`. After correcting the problem, delete the incomplete output for that sample and run it again.

**5. Can this notebook be used with different tissues?**  
Yes. Set `USE_SAFE_LIST=False`, or provide a safe list appropriate for the target tissue.